#   Iterative Workflows in LangGraph
`Building a blog post automation - generate the post evaluate and if impovement required the again generate if it os approved then loop END`

In [143]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
import os
import operator #reducer

In [144]:
generator_llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4o-mini")
evaluator_llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4o")
optimization_llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4o-mini")

In [145]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [146]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [147]:
#state
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [148]:
def generate_tweet(state: TweetState):
    #prompt
    message = [
        SystemMessage(content='You are funny and clever twitter/X influencer'),
        HumanMessage(content="""
Write a short , original and hilarious tweet on topic : "{state['topic]}".
                     
Rules:
- Do not use question answer format
- Max 280 character
- Use obsrvational humor, irony, sarcasm, or cultural refrences.
- Think in meme logic, punchlines, or relatable takes.
- Use simple daya to day english.                                          
""")
    ]

    #send generator llm
    response = generator_llm.invoke(message).content
    return {'tweet': response, 'tweet_history': [response]}


In [149]:
def evaluate_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    #send evaluation llm
    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}


In [150]:
def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimization_llm.invoke(messages).content
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}



In [151]:
def route_evaluation(state: TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [152]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()



In [153]:
initial_state = {
    'topic': 'Indian Railways',
    'iteration': 1,
    'max_iteration': 5
}

workflow.invoke(initial_state)

{'topic': 'Indian Railways',
 'tweet': '"Traveling on Indian Railways is less about the destination and more about redefining your limits—will you dodge a snack seller or a family reunion? At least there’s help in the next compartment: a group of wannabe life coaches sharing unsolicited advice! 🚂✨ #IndianRailways"',
 'evaluation': 'approved',
 'feedback': 'This tweet skillfully navigates a comedic commentary on the typical experiences of traveling on Indian Railways, maintaining originality by weaving distinct cultural observations like dodging snack sellers and encountering unsolicited advice from wannabe life coaches. It delivers humor in a relatable way that might resonate with frequent railway travelers, offering a nod to chaotic yet charming train journeys in India. Additionally, it effectively avoids question-answer or setup-punchline formats and adheres to the character limit. While not overwhelmingly hilarious, its humorous depiction of common travel quirks has potential for vi